# Notebook 08 — Open Science Practices

**Module:** 18 — Scientific Writing and Open Science  
**Tier:** 2 — Working competence  
**Notebook:** 8 of 12  
**Time estimate:** 60 minutes

> Open science is not just an ethical stance — it is a career advantage.
> A bioRxiv preprint increases your citation count. A Zenodo DOI makes
> your code citable. An ORCID links all your outputs. This notebook
> covers the practical steps.

---
## Step 1 — Why Open Science Matters for This Programme

For Track A (India RA/JRF applications): code on GitHub is evidence of skill.
A public, well-documented repository with real biological data analysis is
worth more than a CV bullet point that says "proficient in Python."

For Track B (European PhD applications): many EMBL, IMPRS, and Wellcome Sanger
PhD programmes explicitly ask for a GitHub URL or a list of public outputs.
A Zenodo-archived repository is citable evidence that you have shipped real work.

---
## Step 2 — The Open Science Stack

| Tool | Purpose | Action |
|------|---------|--------|
| **GitHub** | Version control + public code | Keep this repository public; write good READMEs |
| **Zenodo** | Persistent DOI for code/data snapshots | Connect repo to Zenodo; create a release → auto-DOI |
| **bioRxiv** | Preprint server for biology | Post preprints before (or instead of) journal submission |
| **ORCID** | Persistent researcher identifier | Create one at orcid.org; add to every paper's author list |
| **OSF** | Open Science Framework — data/materials sharing | Use for sharing analysis code + data together |
| **Protocols.io** | Wet lab protocol sharing | Relevant if transitioning to experimental work |

---
## Step 3 — Preprints: bioRxiv and medRxiv

**Why post a preprint?**
- Establishes priority immediately (timestamp is public)
- Gets community feedback before peer review
- Increases visibility — many readers find papers on bioRxiv before the journal
- Is now acceptable at all major biology journals (Nature, Cell, PLOS, eLife, etc.)

**bioRxiv moderation:** papers are checked for format, not content. Accepted within
24–48 hours. Each version gets its own DOI; the canonical URL points to the latest.

**What to post:** any complete analysis that produces a result — even a curriculum
mini-project that uses real biological data, is reproducible, and has real conclusions.
Not required, but the bar is lower than you think.

---
## Step 4 — Data Management

**FAIR data (Wilkinson et al. 2016):**
- Raw data is archived at a specialist repository (GEO for transcriptomics,
  EBI for genomics, PDB for structures) — not just on GitHub.
- Processed data and code are archived at Zenodo (for a stable DOI).
- README in the `datasets/` folder describes exactly what each file is.

**What never goes into this public repo:**
- Full copyrighted paper PDFs (see project constitution §9)
- Raw human genomic data (GDPR; check consent; use controlled-access repos)
- Large binary files (> 50 MB) — use Git LFS or Zenodo instead
- API keys, credentials, or tokens

In [ ]:
import re
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---- Repository open-science readiness audit ----

def audit_repo_openness(readme_text, has_license, has_zenodo_badge,
                          has_orcid, has_github_url, has_biorxiv_link,
                          datasets_have_accessions, code_runs_from_scratch):
    """
    Score a repository's open science readiness on 10 criteria.
    """
    scores = {
        'README exists and describes the project':  len(readme_text) > 200,
        'One-command install/reproduce documented':  any(kw in readme_text.lower()
                                                          for kw in ['pip install', 'conda env',
                                                                       'make', 'reproduce']),
        'License file present':                      has_license,
        'Zenodo DOI badge in README':                has_zenodo_badge,
        'Author ORCID listed':                       has_orcid,
        'GitHub URL in paper/repo':                  has_github_url,
        'bioRxiv link (if paper exists)':            has_biorxiv_link,
        'Datasets have accession numbers':           datasets_have_accessions,
        'Code runs from clean clone':                code_runs_from_scratch,
        'Data and code availability statement':      any(kw in readme_text.lower()
                                                          for kw in ['data availability',
                                                                       'code availability',
                                                                       'availability statement']),
    }
    return scores

# Simulate two repos
README_WEAK = """
# My project
This is my code for the bioinformatics project.
Run main.py to get results.
"""

README_STRONG = """
# DeepBind-v2: TF Binding Prediction with Convolutional Networks

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.XXXXXXX.svg)](https://doi.org/10.5281/zenodo.XXXXXXX)

## Overview
This repository contains code and data for DeepBind-v2, a CNN for transcription
factor binding site prediction trained on ENCODE ChIP-seq data.

## Reproduce all results
```bash
conda env create -f environment.yml
conda activate deepbind-v2
python reproduce.py  # runs all experiments from scratch
```

## Data availability
Raw ChIP-seq data: ENCODE (accession ENCSR000AKF). Processed sequences: zenodo.org/XXXXXXX.
Code availability: this repository (MIT license). All figures reproducible from above command.
"""

scores_weak   = audit_repo_openness(README_WEAK,   False, False, False, False, False, False, False)
scores_strong = audit_repo_openness(README_STRONG, True,  True,  True,  True,  True,  True,  True)

print('=== Repository open science audit ===')
print(f'{"Criterion":<45} {"Weak":>6} {"Strong":>8}')
print('-' * 62)
for criterion in scores_weak:
    w = '✓' if scores_weak[criterion]   else '✗'
    s = '✓' if scores_strong[criterion] else '✗'
    print(f'{criterion:<45} {w:>6} {s:>8}')

weak_score   = sum(scores_weak.values())
strong_score = sum(scores_strong.values())
n_total = len(scores_weak)
print(f'\nTotal: Weak {weak_score}/{n_total}, Strong {strong_score}/{n_total}')

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# Panel A: Open science stack diagram
ax = axes[0]
ax.axis('off')
stack = [
    ('ORCID',      '#4e79a7', 'Links all your outputs\norcid.org — create one now'),
    ('GitHub',     '#f28e2b', 'Public code + version history\nPublic repo = Track A/B artifact'),
    ('Zenodo',     '#59a14f', 'Persistent DOI for snapshots\nConnect to GitHub → auto-DOI'),
    ('bioRxiv',    '#e15759', 'Preprint before journal\nEstablishes priority + citations'),
    ('Data repos', '#76b7b2', 'GEO / EBI / PDB / Zenodo\nFor raw and processed data'),
]
for i, (name, color, desc) in enumerate(stack):
    y = 0.90 - i * 0.18
    ax.add_patch(mpatches.FancyBboxPatch((0.05, y-0.08), 0.9, 0.14,
                                           boxstyle='round,pad=0.02', facecolor=color,
                                           alpha=0.3, transform=ax.transAxes))
    ax.text(0.5, y + 0.01, name, transform=ax.transAxes, fontsize=10,
              fontweight='bold', ha='center', va='center')
    ax.text(0.5, y - 0.04, desc, transform=ax.transAxes, fontsize=8,
              ha='center', va='center', color='#333')
ax.set_title('A. Open science tool stack')

# Panel B: Repository readiness scores
ax = axes[1]
criteria_short = ['README\ndescriptive', 'One-command\nreproduce', 'License', 'Zenodo\nDOI',
                    'ORCID', 'GitHub\nURL', 'bioRxiv', 'Data\naccessions',
                    'Clean\nclone', 'Availability\nstatement']
x = np.arange(len(criteria_short))
w = 0.35
weak_vals   = [int(v) for v in scores_weak.values()]
strong_vals = [int(v) for v in scores_strong.values()]
ax.bar(x - w/2, weak_vals,   w, label='Weak repo',   color='tomato',    edgecolor='black', alpha=0.8)
ax.bar(x + w/2, strong_vals, w, label='Strong repo', color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(criteria_short, fontsize=7, rotation=30, ha='right')
ax.set_ylim(0, 1.3); ax.set_ylabel('Criterion met (0 or 1)')
ax.set_title('B. Repository openness audit\n(10 criteria)')
ax.legend(fontsize=9)

# Panel C: Preprint impact evidence
ax = axes[2]
# Schematic: preprint citation accumulation vs journal-only
months = np.arange(0, 36)
journal_citations = np.where(months < 12, 0, 2 * (months - 12)**0.7)
biorxiv_citations = 0.5 * months**0.9
ax.plot(months, journal_citations, color='tomato',    lw=2, label='Journal only\n(peer review = ~12 months)')
ax.plot(months, biorxiv_citations, color='steelblue', lw=2, label='Preprint + Journal')
ax.axvline(12, color='grey', ls='--', lw=1, label='Average peer review time')
ax.fill_between(months, journal_citations, biorxiv_citations,
                  where=(biorxiv_citations > journal_citations),
                  alpha=0.2, color='steelblue', label='Citation advantage')
ax.set_xlabel('Months after submission')
ax.set_ylabel('Cumulative citations (schematic)')
ax.set_title('C. Preprint citation advantage\n(schematic; Rxivist 2020 data)')
ax.legend(fontsize=8)

plt.suptitle('Module 18 NB08: Open Science Practices', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('open_science_practices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

See E08 in `exercises/README.md`:
1. Run `audit_repo_openness()` on this curriculum repository's README.
   Report the score.
2. Write a 50-word data and code availability statement for any prior
   module mini-project, following the format in README_STRONG above.

---
## Step 10 — Quiz

1. What is an ORCID and why should every researcher have one?
2. What is the difference between GitHub and Zenodo? When do you need Zenodo?
3. Name three things that must never go into a public GitHub repository.
4. What is a preprint? What are the two main preprint servers for biology
   and medicine? What is their relationship to peer-reviewed publication?

---
## Step 12 — Reflection

> *[Run the repository audit on the current state of your computational-biology
> repo. What are the top 3 missing criteria? Write a plan (one sentence each)
> for how you would address each one before Month 6.]*

---
*Next: `09_peer_review.ipynb`*